In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Features: 1000 muestras x 270 columnas (CSI aplanado 30x3x3)
X_train = pd.read_csv('Train_features.csv', header=None).values
X_test  = pd.read_csv('Test_features.csv',  header=None).values

# Labels: enteros del 1 al 5 (wave, push, crouch, sitdown, bend)
y_train = pd.read_csv('Train_labels.csv', header=None).values.flatten()
y_test  = pd.read_csv('Test_labels.csv',  header=None).values.flatten()

# Skeleton points: 54 columnas (18 x + 18 y + 18 confidence)
skel_train = pd.read_csv('Train_skelletonpoints.csv', header=None).values
skel_test  = pd.read_csv('Test_skelletonpoints.csv',  header=None).values

POSE_NAMES = {1: 'wave', 2: 'push', 3: 'crouch', 4: 'sitdown', 5: 'bend'}

print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   |  y_test:  {y_test.shape}')
print(f'Clases disponibles: {np.unique(y_train)}')

In [ ]:
# Conexiones entre los 18 keypoints (índice 0-based)
# nose(0), neck(1), r_shoulder(2), r_elbow(3), r_wrist(4),
# l_shoulder(5), l_elbow(6), l_wrist(7), r_hip(8), r_knee(9), r_ankle(10),
# l_hip(11), l_knee(12), l_ankle(13), r_eye(14), l_eye(15), r_ear(16), l_ear(17)
CONNECTIONS = [
    (0, 1),   # Nose - Neck
    (1, 2),   # Neck - Right Shoulder
    (2, 3),   # Right Shoulder - Right Elbow
    (3, 4),   # Right Elbow - Right Wrist
    (1, 5),   # Neck - Left Shoulder
    (5, 6),   # Left Shoulder - Left Elbow
    (6, 7),   # Left Elbow - Left Wrist
    (1, 8),   # Neck - Right Hip
    (8, 9),   # Right Hip - Right Knee
    (9, 10),  # Right Knee - Right Ankle
    (1, 11),  # Neck - Left Hip
    (11, 12), # Left Hip - Left Knee
    (12, 13), # Left Knee - Left Ankle
    (0, 14),  # Nose - Right Eye
    (0, 15),  # Nose - Left Eye
    (14, 16), # Right Eye - Right Ear
    (15, 17), # Left Eye - Left Ear
    (2, 5),   # Right Shoulder - Left Shoulder
    (8, 11),  # Right Hip - Left Hip
]

def plot_skeleton(skeleton_row, label, ax=None):
    """Dibuja el esqueleto de una muestra con su etiqueta."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 6))

    x_coords = skeleton_row[:18]
    y_coords = skeleton_row[18:36]

    for (i, j) in CONNECTIONS:
        ax.plot([y_coords[i], y_coords[j]],
                [x_coords[i], x_coords[j]],
                'b-', linewidth=1.5)

    ax.scatter(y_coords, x_coords, c='red', s=30, zorder=3)
    ax.set_title(f'Pose = {POSE_NAMES[label]}', fontsize=13)
    ax.set_xlabel('Y-coordinate')
    ax.set_ylabel('X-coordinate')
    ax.invert_yaxis()


# Mostrar un ejemplo de cada pose
fig, axes = plt.subplots(1, 5, figsize=(18, 6))
fig.suptitle('Ejemplo de cada pose', fontsize=15)

for pose_id, ax in zip(range(1, 6), axes):
    idx = np.where(y_train == pose_id)[0][0]
    plot_skeleton(skel_train[idx], y_train[idx], ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
def plot_csi_amplitude(features_row, label, ax=None):
    """Plot de la amplitud CSI media por subcarrier para una muestra."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4))

    # Reshape: (270,) -> (30 subcarriers, 3 TX, 3 RX)
    csi_matrix = features_row.reshape(30, 3, 3)

    # Media sobre las 9 combinaciones de antenas -> (30,)
    csi_mean = csi_matrix.mean(axis=(1, 2))

    ax.plot(range(1, 31), csi_mean, marker='o', markersize=4, linewidth=1.5)
    ax.set_title(f'Amplitud CSI media — Pose: {POSE_NAMES[label]}')
    ax.set_xlabel('Subcarrier')
    ax.set_ylabel('Amplitud media')
    ax.set_xticks(range(1, 31))
    ax.grid(True, alpha=0.3)


# Un ejemplo por pose
fig, axes = plt.subplots(5, 1, figsize=(10, 18))
fig.suptitle('Amplitud CSI media por subcarrier (un ejemplo por pose)', fontsize=14)

for pose_id, ax in zip(range(1, 6), axes):
    idx = np.where(y_train == pose_id)[0][0]
    plot_csi_amplitude(X_train[idx], y_train[idx], ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot por subcarrier para cada clase
fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=True)
fig.suptitle('Distribución de amplitud CSI por subcarrier y clase', fontsize=14)

colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

for pose_id, ax, color in zip(range(1, 6), axes, colors):
    mask = y_train == pose_id
    samples = X_train[mask]  # (n_samples, 270)

    # Media sobre las 9 antenas -> (n_samples, 30)
    csi_reshaped = samples.reshape(len(samples), 30, 9).mean(axis=2)

    ax.boxplot(csi_reshaped, positions=range(1, 31),
               patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='black', linewidth=2),
               whiskerprops=dict(linewidth=1),
               flierprops=dict(marker='.', markersize=2, alpha=0.3),
               widths=0.6)

    ax.set_title(f'{POSE_NAMES[pose_id]}\n(n={mask.sum()})', fontsize=12)
    ax.set_xlabel('Subcarrier')
    if pose_id == 1:
        ax.set_ylabel('Amplitud CSI')
    ax.set_xticks(range(1, 31, 5))
    ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Vista comparativa: todas las clases superpuestas (media ± std)
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

for pose_id, color in zip(range(1, 6), colors):
    mask = y_train == pose_id
    samples = X_train[mask].reshape(mask.sum(), 30, 9).mean(axis=2)  # (n, 30)
    mean_per_sub = samples.mean(axis=0)
    std_per_sub  = samples.std(axis=0)

    ax.plot(range(1, 31), mean_per_sub, color=color,
            label=POSE_NAMES[pose_id], linewidth=2)
    ax.fill_between(range(1, 31),
                    mean_per_sub - std_per_sub,
                    mean_per_sub + std_per_sub,
                    color=color, alpha=0.15)

ax.set_title('Media ± std de amplitud CSI por subcarrier y clase', fontsize=13)
ax.set_xlabel('Subcarrier')
ax.set_ylabel('Amplitud CSI')
ax.set_xticks(range(1, 31))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()